# 🍊🍌 Perceptrón Online: Clasificador de Frutas

## El problema

Un agricultor tiene una **banda transportadora** por la que pasan naranjas y plátanos mezclados. Instala un sensor que mide dos características de cada fruta:

| Medición | Variable | Descripción |
|----------|----------|-------------|
| Redondez | $x_1$   | Más alto → más redonda |
| Color    | $x_2$   | Más alto → más amarillo |

Necesita un **clasificador automático** que aprenda a separar las frutas trazando una línea divisoria en el plano $(x_1, x_2)$.

## El perceptrón

El perceptrón es el clasificador lineal más simple. Aprende una frontera de decisión de la forma:

$$w_1 x_1 + w_2 x_2 + b = 0$$

- Si $\mathbf{w} \cdot \mathbf{x} + b \geq 0$ → predice **plátano** (clase +1)
- Si $\mathbf{w} \cdot \mathbf{x} + b < 0$ → predice **naranja** (clase -1)

### Regla de aprendizaje

Cuando se equivoca en un punto $(\mathbf{x}_i, y_i)$, corrige sus pesos:

$$\mathbf{w} \leftarrow \mathbf{w} + \eta (y_i - \hat{y}_i) \mathbf{x}_i$$
$$b \leftarrow b + \eta (y_i - \hat{y}_i)$$

donde $\eta$ es la tasa de aprendizaje (*learning rate*).

### Propiedad geométrica clave

El vector $\mathbf{w}$ es **siempre perpendicular** a la frontera de decisión. Apunta hacia la región donde el perceptrón predice clase +1 (plátanos).

---

En este notebook verás paso a paso cómo el perceptrón **aprende** a separar las frutas, corrigiendo la línea cada vez que se equivoca.

## 1. Importar librerías

Solo usamos **numpy** (cálculo numérico) y **matplotlib** (gráficas), ambas incluidas por defecto en Google Colab.

In [ ]:
import numpy as np                        # Álgebra lineal y operaciones numéricas
import matplotlib.pyplot as plt            # Gráficas 2D
from matplotlib.patches import Patch       # Elementos personalizados para la leyenda
from matplotlib import animation           # Animaciones cuadro por cuadro
from IPython.display import HTML, display  # Mostrar el video HTML5 en Colab
import warnings
warnings.filterwarnings('ignore')          # Ocultar warnings cosméticos de matplotlib

print('Librerías cargadas correctamente')

## 2. Datos del sensor de frutas

Definimos las mediciones del sensor para 10 naranjas y 10 plátanos.

Los datos están **cerca entre sí** a propósito: si estuvieran en esquinas opuestas, el perceptrón los separaría en 1 paso y no veríamos nada interesante. Al estar mezclados espacialmente, el aprendizaje necesita **varias correcciones** para encontrar la línea óptima.

In [ ]:
# ──────────────────────────────────────────────────────────────
# Clase -1: NARANJAS 🍊
# Cada fila es una fruta: [redondez, color]
# ──────────────────────────────────────────────────────────────
X_naranja = np.array([
    [-2.0,  1.0],   # naranja 0: muy redonda, algo amarilla
    [-1.5, -1.0],   # naranja 1: redonda, poco amarilla
    [-0.5,  0.5],   # naranja 2: intermedia
    [-1.0, -0.5],   # naranja 3
    [ 0.0,  1.5],   # naranja 4: esta está cerca del borde → difícil
    [-1.5,  0.0],   # naranja 5
    [-0.3, -1.0],   # naranja 6
    [-1.8,  0.5],   # naranja 7
    [ 0.2, -0.3],   # naranja 8: muy cerca de los plátanos → difícil
    [-0.8,  1.2],   # naranja 9
])

# ──────────────────────────────────────────────────────────────
# Clase +1: PLÁTANOS 🍌
# ──────────────────────────────────────────────────────────────
X_platano = np.array([
    [ 1.0, -1.0],   # plátano 0
    [ 0.5,  1.0],   # plátano 1: cerca de las naranjas → difícil
    [ 1.5,  0.5],   # plátano 2
    [ 0.8, -0.5],   # plátano 3
    [ 2.0,  0.0],   # plátano 4
    [ 0.3, -1.5],   # plátano 5
    [ 1.2,  1.2],   # plátano 6
    [ 1.8, -0.8],   # plátano 7
    [ 0.5,  0.2],   # plátano 8: muy cerca de las naranjas → difícil
    [ 1.5, -1.5],   # plátano 9
])

# Unir todo en una sola matriz X y un vector de etiquetas y
X = np.vstack([X_naranja, X_platano])              # Forma: (20, 2)
y = np.array([-1]*len(X_naranja) + [1]*len(X_platano))  # -1 para naranjas, +1 para plátanos
N_naranja = len(X_naranja)                          # Para saber qué fruta es cada índice

print(f'Total de frutas: {len(X)}')
print(f'  🍊 Naranjas: {len(X_naranja)}')
print(f'  🍌 Plátanos: {len(X_platano)}')

## 3. Entrenamiento del perceptrón paso a paso

El algoritmo es **online**: procesa **una fruta a la vez**. Si la clasifica bien, no hace nada. Si se equivoca, ajusta $\mathbf{w}$ y $b$.

Guardamos el estado completo después de cada corrección para poder visualizarlo después.

### Parámetros elegidos
- **Peso inicial** $\mathbf{w}_0 = (-0.3, -0.8)$: apunta en dirección equivocada a propósito
- **Sesgo inicial** $b_0 = 0.3$: desplaza la frontera lejos de la posición correcta
- **Learning rate** $\eta = 0.15$: calibrado para que converja en exactamente **8 correcciones**

In [ ]:
def entrenar_perceptron(X, y, w_init, b_init, lr=0.15):
    """
    Entrena un perceptrón y guarda el historial completo.

    Parámetros:
        X       : array (N, 2) con las mediciones del sensor
        y       : array (N,) con las etiquetas (-1 naranja, +1 plátano)
        w_init  : array (2,) pesos iniciales
        b_init  : float, sesgo inicial
        lr      : float, tasa de aprendizaje (eta)

    Retorna:
        history : lista de diccionarios, uno por cada estado guardado
    """

    # ── Inicialización ──
    w = w_init.copy()     # Copiar para no modificar el original
    b = b_init

    # Guardar estado inicial (antes de cualquier aprendizaje)
    history = [{
        'w': w.copy(),
        'b': b,
        'err': None,          # Ningún punto causó error todavía
        'desc': 'Inicio: el sensor no sabe separar las frutas'
    }]

    # ── Bucle de entrenamiento ──
    # Repetimos épocas hasta que no haya errores (convergencia)
    for epoch in range(30):
        errors = 0

        # Recorrer CADA fruta en orden
        for i in range(len(X)):

            # --- Paso 1: Calcular la predicción ---
            # activación = w · x + b  (producto punto + sesgo)
            activacion = np.dot(w, X[i]) + b

            # Regla de decisión: ≥ 0 → plátano (+1), < 0 → naranja (-1)
            pred = 1 if activacion >= 0 else -1

            # --- Paso 2: ¿Se equivocó? ---
            if pred != y[i]:
                errors += 1

                # --- Paso 3: Corregir pesos ---
                # delta = η * (y_real - y_predicho)
                # Como y y pred son -1 o +1:
                #   Si y=+1 y pred=-1 → delta = η*(+1-(-1)) = +2η (empuja w hacia x)
                #   Si y=-1 y pred=+1 → delta = η*(-1-(+1)) = -2η (aleja w de x)
                delta = lr * (y[i] - pred)

                # Actualizar w y b
                w = w + delta * X[i]   # w nuevo = w viejo + δ·x
                b = b + delta          # b nuevo = b viejo + δ

                # --- Guardar para la animación ---
                # Determinar nombres de frutas para la descripción
                if i < N_naranja:
                    fruta_real, fruta_pred = 'naranja', ('platano' if pred==1 else 'naranja')
                else:
                    fruta_real, fruta_pred = 'platano', ('naranja' if pred==-1 else 'platano')

                history.append({
                    'w': w.copy(),
                    'b': b,
                    'err': i,
                    'desc': f'Ep.{epoch}: confundio {fruta_real} con {fruta_pred} -> corrige'
                })

        # --- ¿Convergió? (ningún error en toda la época) ---
        if errors == 0:
            history.append({
                'w': w.copy(), 'b': b, 'err': None,
                'desc': f'Convergencia! Epoca {epoch}: todas las frutas bien clasificadas'
            })
            break  # Ya no hay nada que aprender

    return history


# ── Ejecutar entrenamiento ──
history = entrenar_perceptron(
    X, y,
    w_init = np.array([-0.3, -0.8]),  # Dirección inicial muy mala
    b_init = 0.3,                      # Sesgo inicial desplazado
    lr     = 0.15                      # Tasa de aprendizaje
)

# ── Resumen ──
print(f'Resultado del entrenamiento:')
print(f'  Total de pasos guardados: {len(history)}')
print(f'  - 1 estado inicial')
print(f'  - {len(history)-2} correcciones')
print(f'  - 1 convergencia final')
print()
print('Detalle de cada paso:')
for i, h in enumerate(history):
    w1, w2 = h['w']
    print(f"  [{i+1:>2}] w=({w1:+.2f}, {w2:+.2f})  b={h['b']:+.2f}  | {h['desc']}")

## 4. Función de visualización

Esta función dibuja **un paso** del entrenamiento. Muestra:

- **Regiones coloreadas**: zona naranja (el sensor predice naranja) y zona amarilla (predice plátano)
- **Línea roja**: la frontera de decisión $\mathbf{w} \cdot \mathbf{x} + b = 0$
- **Flecha azul**: el vector $\mathbf{w}$, siempre perpendicular a la frontera
- **Halo rojo**: la fruta que fue mal clasificada y causó la corrección
- **Barra de precisión**: qué porcentaje de frutas están bien clasificadas

In [ ]:
# ── Límites del gráfico (fijos para toda la animación) ──
MARGEN = 1.5
XMIN, XMAX = X[:,0].min() - MARGEN, X[:,0].max() + MARGEN
YMIN, YMAX = X[:,1].min() - MARGEN, X[:,1].max() + MARGEN


def dibujar(ax, idx):
    """
    Dibuja el estado del perceptrón en el paso 'idx'.

    Parámetros:
        ax  : eje de matplotlib donde dibujar
        idx : índice del paso en el historial (0 = inicio)
    """
    ax.clear()  # Limpiar el eje para redibujar desde cero

    # ── Recuperar estado de este paso ──
    estado = history[idx]
    w  = estado['w']       # Vector de pesos (2D)
    b  = estado['b']       # Sesgo
    err = estado['err']    # Índice de la fruta que causó error (o None)
    w1, w2 = w             # Componentes individuales

    # ── Configurar ejes ──
    ax.set_aspect('equal')               # Misma escala en ambos ejes
    ax.set_xlim(XMIN, XMAX)             # (para que la perpendicularidad se vea real)
    ax.set_ylim(YMIN, YMAX)
    ax.grid(True, alpha=0.2)
    ax.axhline(0, color='#bbb', lw=0.4) # Líneas de referencia en x=0, y=0
    ax.axvline(0, color='#bbb', lw=0.4)

    # ─────────────────────────────────────────────────────
    # REGIONES COLOREADAS
    # Evaluamos w·x + b en una cuadrícula densa.
    # Donde es < 0 → zona naranja. Donde es ≥ 0 → zona plátano.
    # ─────────────────────────────────────────────────────
    xx, yy = np.meshgrid(
        np.linspace(XMIN, XMAX, 300),
        np.linspace(YMIN, YMAX, 300)
    )
    zz = w1 * xx + w2 * yy + b  # Activación en cada punto de la cuadrícula

    # contourf rellena: primer color donde zz<0, segundo donde zz≥0
    # ╔═══════════════════════════════════════════════════╗
    # ║  AQUÍ SE CAMBIAN LOS COLORES DE LAS REGIONES    ║
    # ║  colors=[zona_naranja, zona_platano]             ║
    # ║  alpha=intensidad (0.0 a 1.0)                    ║
    # ╚═══════════════════════════════════════════════════╝
    ax.contourf(xx, yy, zz, levels=[-1e10, 0, 1e10],
                colors=['#ff9800', '#fdd835'], alpha=0.6)

    # La línea donde zz=0 es la frontera de decisión
    ax.contour(xx, yy, zz, levels=[0], colors=['#d32f2f'], linewidths=3)

    # ─────────────────────────────────────────────────────
    # FRUTAS (puntos de datos)
    # ─────────────────────────────────────────────────────
    for j in range(len(X)):
        es_naranja = (y[j] == -1)
        color = '#ff9800' if es_naranja else '#fdd835'    # Naranja o amarillo
        marcador = 'o' if es_naranja else 's'             # Círculo o cuadrado
        tamano, borde, grosor = 130, 'black', 1.5         # Tamaño por defecto

        # Si esta fruta causó el error en este paso, resaltarla
        if j == err:
            # Halo rojo semitransparente detrás
            ax.scatter(X[j,0], X[j,1], s=550, c='red', alpha=0.25, zorder=1)
            tamano, borde, grosor = 200, '#d32f2f', 2.5

        # Dibujar la fruta
        ax.scatter(X[j,0], X[j,1], c=color, s=tamano, edgecolors=borde,
                   lw=grosor, zorder=3, marker=marcador)

        # Emoji encima del punto (se renderiza bien en Colab)
        emoji = '\U0001F34A' if es_naranja else '\U0001F34C'  # 🍊 o 🍌
        ax.text(X[j,0], X[j,1], emoji, ha='center', va='center',
                fontsize=14 if j==err else 10, zorder=4)

    # ─────────────────────────────────────────────────────
    # VECTOR W (perpendicular a la frontera)
    # Lo normalizamos para que siempre tenga el mismo largo visual.
    # Geométricamente, w apunta hacia la región de clase +1.
    # ─────────────────────────────────────────────────────
    norma_w = np.linalg.norm(w)          # ||w|| = sqrt(w1² + w2²)
    if norma_w > 1e-6:                   # Evitar división por cero
        w_unitario = w / norma_w          # Dirección de w (largo 1)
        largo_flecha = 2.0                # Largo visual constante
        punta = w_unitario * largo_flecha

        # Flecha desde el origen hasta punta
        ax.annotate('', xy=(punta[0], punta[1]), xytext=(0, 0),
                    arrowprops=dict(arrowstyle='->', color='#1565c0', lw=3))

        # Etiqueta "w" al final de la flecha
        ax.text(punta[0]*1.18, punta[1]*1.18, 'w', fontsize=15,
                color='#1565c0', fontweight='bold', ha='center')

    # ─────────────────────────────────────────────────────
    # INFORMACIÓN EN PANTALLA
    # ─────────────────────────────────────────────────────
    # Contar cuántas frutas están bien clasificadas con estos w, b
    correctas = sum(
        1 for j in range(len(X))
        if (1 if np.dot(w, X[j]) + b >= 0 else -1) == y[j]
    )
    porcentaje = 100 * correctas / len(X)

    # Barra visual de progreso
    n_llenos = int(porcentaje / 5)                    # Cada bloque = 5%
    barra = chr(9608)*n_llenos + chr(9617)*(20-n_llenos)  # █░
    estado_txt = 'SEPARADAS!' if correctas==len(X) else f'{correctas}/{len(X)}'

    # Texto con info del paso
    info = (f'Paso {idx+1}/{len(history)}   Precision: {estado_txt} ({porcentaje:.0f}%)\n'
            f'[{barra}]\n'
            f'w = ({w1:+.2f}, {w2:+.2f})   b = {b:+.2f}   ||w|| = {norma_w:.2f}')

    ax.text(0.02, 0.02, info, transform=ax.transAxes, fontsize=9,
            va='bottom', family='monospace',
            bbox=dict(boxstyle='round,pad=0.5', fc='white', ec='#ccc', alpha=0.9))

    # Título del paso
    ax.set_title(estado['desc'], fontsize=12, fontweight='bold', pad=10,
                 color='#2e7d32' if correctas==len(X) else '#333')
    ax.set_xlabel('x1 (redondez)', fontsize=11)
    ax.set_ylabel('x2 (color)', fontsize=11)

    # Leyenda
    leyenda = [
        plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='#ff9800',
                   markeredgecolor='k', markersize=10, label='Naranja (clase -1)'),
        plt.Line2D([0],[0], marker='s', color='w', markerfacecolor='#fdd835',
                   markeredgecolor='k', markersize=10, label='Platano (clase +1)'),
        plt.Line2D([0],[0], color='#d32f2f', lw=2.5, label='Frontera (w*x+b=0)'),
        plt.Line2D([0],[0], color='#1565c0', lw=2.5, label='Vector w (perp.)'),
    ]
    ax.legend(handles=leyenda, loc='upper left', fontsize=8, framealpha=0.9)


print('Funcion de visualizacion definida')

## 5. Animación

Generamos un video HTML5 que se reproduce **directamente en Colab** con controles de play/pause. Cada cuadro muestra un paso del entrenamiento (2 segundos por paso).

Esto es más confiable que `ipywidgets` en Colab porque no depende del kernel para re-renderizar.

In [ ]:
# ── Crear figura (sin mostrarla todavía) ──
fig, ax = plt.subplots(figsize=(9, 8))
plt.close(fig)  # Evitar que se muestre un frame estático

# ── Crear animación ──
# FuncAnimation llama a 'dibujar' una vez por cada frame.
# frames = número total de pasos
# interval = milisegundos entre frames (2000 = 2 segundos)
anim = animation.FuncAnimation(
    fig,
    func     = lambda frame: dibujar(ax, frame),  # Qué dibujar en cada frame
    frames   = len(history),                        # Cuántos frames
    interval = 2000,                                # 2 seg por paso
    repeat   = True                                 # Repetir al terminar
)

# ── Convertir a video HTML5 y mostrar ──
display(HTML(f"""
<div style="text-align:center; margin:10px 0">
  <h3 style="color:#333">
    Perceptron Online: Clasificador de Frutas
  </h3>
  <p style="color:#555; font-size:14px; max-width:620px; margin:8px auto">
    El agricultor entrena su sensor para separar naranjas y platanos.<br>
    Cada vez que el sensor confunde una fruta, corrige sus pesos.
  </p>
  {anim.to_html5_video()}
  <p style="margin-top:12px; font-size:13px; color:#888">
    Punto con halo <b style="color:red">rojo</b> = fruta que el sensor confundio
    <br>
    Zona <span style="background:#ff9800;color:white;padding:2px 8px;border-radius:3px">naranja</span>
    = predice naranja &nbsp;|&nbsp;
    Zona <span style="background:#fdd835;padding:2px 8px;border-radius:3px">amarilla</span>
    = predice platano
  </p>
</div>
"""))

print('\nQue observar:')
print('  1. Al inicio la frontera esta mal puesta (confunde muchas frutas)')
print('  2. Cada error corrige w y b: la linea rota y se desplaza')
print('  3. El vector w (azul) SIEMPRE es perpendicular a la frontera (roja)')
print('  4. La barra de precision sube con cada correccion')
print('  5. Al final: naranjas de un lado, platanos del otro')
print()
print('Formula de correccion:')
print('  w <- w + eta * (y_real - y_pred) * x')
print('  b <- b + eta * (y_real - y_pred)')

## 6. (Opcional) Exploración manual con slider

Si prefieres avanzar paso a paso a tu ritmo, descomenta y ejecuta la siguiente celda. Usa el slider para ir hacia adelante y atrás.

In [ ]:
# Descomenta las siguientes líneas para usar el slider:

# from ipywidgets import interact, IntSlider
#
# @interact(paso=IntSlider(min=0, max=len(history)-1, step=1, value=0,
#           description='Paso:', continuous_update=False,
#           layout={'width': '80%'}))
# def mostrar(paso):
#     fig2, ax2 = plt.subplots(figsize=(9, 8))
#     dibujar(ax2, paso)
#     plt.tight_layout()
#     plt.show()

## 7. Resumen

| Concepto | En este ejemplo |
|----------|----------------|
| **Modelo** | Perceptrón (clasificador lineal binario) |
| **Frontera** | Línea recta $w_1 x_1 + w_2 x_2 + b = 0$ |
| **Vector w** | Perpendicular a la frontera, apunta hacia clase +1 |
| **Aprendizaje** | Online: corrige solo cuando se equivoca |
| **Convergencia** | Garantizada si los datos son linealmente separables |
| **Limitación** | No puede separar datos que NO sean linealmente separables (ej: XOR) |

### Para experimentar

Prueba cambiar en la celda de entrenamiento:
- `w_init`: otro vector inicial → ¿tarda más o menos?
- `lr`: learning rate más alto (0.5) o más bajo (0.01) → ¿qué pasa?
- Los datos en `X_naranja` y `X_platano`: ¿puedes hacer que NO converja? (pista: haz que sean no separables linealmente)